# Demo de inferencia · pronóstico a 24 h del SEIN
Carga los **pesos entrenados** (`results_v2/<experimento>/modelo_<semilla>.pt`) y pronostica las 24 horas de un día elegido, comparando con lo ejecutado, Ridge y la programación diaria del COES. No entrena nada: solo inferencia (tarda segundos). Requiere haber ejecutado antes el cuaderno B2.

In [ ]:
from pathlib import Path
import os, sys, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_ = Path("/content/drive/MyDrive/MIA07 REDES NEURONALES Y APRENDIZAJE PROFUNDO")
    PROYECTO = next((c for c in (RAIZ_ / "23 PROYECTO FINAL" / "02_Repositorio_Codigo", RAIZ_ / "23 PROYECTO FINAL" / "04 PROYECTO COES DEMANDA",
                                 RAIZ_ / "24 PREVIOS PROYECTO FINAL" / "04 PROYECTO COES DEMANDA") if (c / "src" / "exp_v2.py").is_file()),
                    RAIZ_ / "23 PROYECTO FINAL" / "02_Repositorio_Codigo")
except ImportError:
    PROYECTO = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
assert (PROYECTO / "src" / "demo_v2.py").is_file(), f"No encuentro src/demo_v2.py en {PROYECTO}"
if importlib.util.find_spec("holidays") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "holidays"], check=True)
print("Proyecto:", PROYECTO)

# Datos recientes del COES: se guardan fuera de la carpeta del repositorio (los términos del COES no permiten redistribuirlos)
ACTUALIZAR_DATOS = True        # True: descarga datos hasta hoy (~1 min) para poder elegir fechas cercanas
CSV = Path("/content/coes_datos/coes_demanda_limpio.csv") if Path("/content").is_dir() else PROYECTO / "data" / "coes_demanda_limpio.csv"
if ACTUALIZAR_DATOS or not CSV.is_file():
    subprocess.run([sys.executable, str(PROYECTO / "coes_descarga.py"), "--salida", str(CSV.parent)], check=True, cwd=PROYECTO)
os.environ["COES_CSV"] = str(CSV)
print("Datos:", CSV)


In [ ]:
sys.path.insert(0, str(PROYECTO / "src"))
import numpy as np
from demo_v2 import pronosticar, graficar, ultimo_dia_disponible

FECHA = "auto"                  # "auto" = el último día completo disponible; o escribe una fecha AAAA-MM-DD (p. ej. "2026-11-04")
if FECHA == "auto":
    FECHA = ultimo_dia_disponible()
print("Día pronosticado:", FECHA)
MODELO, SEMILLA = "gru_A1B1C0", 17
tabla = pronosticar(FECHA, "results_v2", MODELO, SEMILLA)
display(tabla)
for c in ("ridge_mw", "gru_mw", "coes_diaria_mw"):
    print(f"MAE {c:16s}: {np.abs(tabla[c] - tabla.real_mw).mean():6.1f} MW")
graficar(tabla, FECHA)
